# Workspace Analysis — Interactive Notebook

Interactive analysis of pre-computed Monte Carlo FK workspace data.

**Workflow:**
1. Set parameters (robot, workspace bounds, voxel size) in the cell below.
2. Run all cells to load data, compute statistics, and generate plots.
3. Use the *Densify* button at the bottom to run additional samples
   (appended to existing data).

**References:**
- [1] Yoshikawa (1985) — Manipulability Index
- [2] Salisbury & Craig (1982) — Inverse Condition Number
- [3] Klein (2023) — Desired workspace geometry
- [4] MathWorks (2024) — Monte Carlo FK workspace analysis

## 1. Parameters

In [ ]:
# ── Robot & data paths ────────────────────────────────────────────────
ROBOT = "kinova"  # "tensegrity" | "ur10e" | "kinova"

# Output directory relative to package root (None = per-robot default).
OUTPUT_DIR: str | None = None


# ── Mount configuration ───────────────────────────────────────────────
# Direction: "down" = ceiling-mounted, "up" = floor/table-mounted.
# Height:   None = per-robot default, or override in metres.
MOUNT_DIRECTION: str | None = None  # None = per-robot default
MOUNT_HEIGHT: float | None = None   # None = per-robot default
# ── Desired workspace bounds [Klein 2023] ─────────────────────────────
WS_MIN = (-0.05, -0.40, 0.80)  # (x_min, y_min, z_min) in metres
WS_MAX = ( 0.35,  0.95, 1.30)  # (x_max, y_max, z_max) in metres

# ── Visualisation ─────────────────────────────────────────────────────
VOXEL_SIZE = 0.02          # Voxel edge length (metres)
SLICE_THICKNESS = 0.05     # 2-D cross-section thickness (metres)

# ── Sampling settings ──────────────────────────────────────────────────
DENSIFY_SAMPLES = 200_000
DENSIFY_NUM_ENVS = 4096
DENSIFY_TIMEOUT_S = 600

## 2. Setup & Data Loading

In [ ]:
from pathlib import Path

import numpy as np

from workspace_analysis_helper import (
    DEFAULT_MOUNT_DIRECTION,
    DEFAULT_OUTPUT_DIRS,
    ROBOT_DISPLAY_NAMES,
    display_comparison_table,
    display_statistics_table,
    draw_cross_section_figure,
    interactive_3d_density,
    interactive_3d_scatter,
    load_workspace_data,
)

PACKAGE_ROOT = Path("../..")
ws_min = np.array(WS_MIN)
ws_max = np.array(WS_MAX)
robot_display = ROBOT_DISPLAY_NAMES.get(ROBOT, ROBOT)
output_dir = PACKAGE_ROOT / (OUTPUT_DIR or DEFAULT_OUTPUT_DIRS[ROBOT])

mount_direction = MOUNT_DIRECTION or DEFAULT_MOUNT_DIRECTION[ROBOT]

positions, yoshikawa, condition = load_workspace_data(output_dir)

print(f"Robot       : {robot_display}")
print(f"Direction   : {mount_direction}")
print(f"Directory   : {output_dir.resolve()}")
print(f"Samples     : {len(positions):,}")

## 3. Statistics Summary

In [ ]:
display_statistics_table(
    positions, yoshikawa, condition, robot_display,
    ws_min=ws_min, ws_max=ws_max, voxel_size=VOXEL_SIZE,
)

## 4. Interactive 3-D Workspace Visualisation

Voxelised 3-D scatter using Plotly — rotate, zoom, and pan with the mouse.

In [ ]:
interactive_3d_scatter(
    positions, yoshikawa, VOXEL_SIZE,
    title=f"{robot_display} — Yoshikawa Manipulability",
    colorbar_label="Yoshikawa w(q)",
    colorscale="Plasma",
    ws_min=ws_min, ws_max=ws_max,
)

In [ ]:
interactive_3d_scatter(
    positions, condition, VOXEL_SIZE,
    title=f"{robot_display} — Inverse Condition Number",
    colorbar_label="\u03c3_min / \u03c3_max",
    colorscale="Inferno",
    ws_min=ws_min, ws_max=ws_max,
)

In [ ]:
interactive_3d_density(
    positions, VOXEL_SIZE,
    title=f"{robot_display} — Reachability Density",
    ws_min=ws_min, ws_max=ws_max,
)

## 5. 2-D Cross-Section Heatmaps

Thin slices through the workspace midpoint along each axis.

In [ ]:
import matplotlib.pyplot as plt

draw_cross_section_figure(
    positions, yoshikawa,
    title=f"{robot_display} — Yoshikawa Manipulability (cross-sections)",
    colorbar_label="Yoshikawa w(q)", cmap="plasma",
    voxel_size=VOXEL_SIZE, slice_thickness=SLICE_THICKNESS,
    ws_min=ws_min, ws_max=ws_max,
    higher_is_better=True,
)
plt.show()

In [ ]:
draw_cross_section_figure(
    positions, condition,
    title=f"{robot_display} — Inverse Condition Number (cross-sections)",
    colorbar_label="\u03c3_min / \u03c3_max", cmap="inferno",
    voxel_size=VOXEL_SIZE, slice_thickness=SLICE_THICKNESS,
    ws_min=ws_min, ws_max=ws_max,
    higher_is_better=True,
)
plt.show()

In [ ]:
draw_cross_section_figure(
    positions, positions[:, 0],
    title=f"{robot_display} — Reachability Density (cross-sections)",
    colorbar_label="log(1+count)", cmap="viridis",
    voxel_size=VOXEL_SIZE, slice_thickness=SLICE_THICKNESS,
    ws_min=ws_min, ws_max=ws_max,
    density_mode=True,
    higher_is_better=True,
)
plt.show()

## 6. Multi-Robot Comparison

In [ ]:
display_comparison_table(PACKAGE_ROOT)

## 7. Densify Measurements

Run the cell below to execute an additional sampling pass.
New samples are **appended** to the existing data via `workspace_sample.py --append`.
Re-run the cells above afterwards to update plots.

In [ ]:
import subprocess

script = PACKAGE_ROOT / "scripts" / "workspace_analysis" / "workspace_sample.py"
cmd = [
    "conda", "run", "-n", "env_isaaclab", 
    "python3", str(script.resolve()),
    "--robot", ROBOT,
    "--num_samples", str(DENSIFY_SAMPLES),
    "--num_envs", str(DENSIFY_NUM_ENVS),
    "--output_dir", str(output_dir.resolve()),
    "--mount_direction", mount_direction,
    "--append", "--headless",
]
if MOUNT_HEIGHT is not None:
    cmd += ["--mount_height", str(MOUNT_HEIGHT)]

print(f"Running densify pass (+{DENSIFY_SAMPLES:,} samples)...")
result = subprocess.run(
    cmd,
    capture_output=True, text=True,
    timeout=DENSIFY_TIMEOUT_S,
    cwd=str(PACKAGE_ROOT.resolve()),
)

if result.returncode != 0:
    print(f"Sampling failed (exit code {result.returncode})")
    print(result.stderr[-2000:] if result.stderr else "(no stderr)")
else:
    print(result.stderr[-2000:] if result.stderr else "(done)")
    positions, yoshikawa, condition = load_workspace_data(output_dir)
    print(f"\nReloaded {len(positions):,} total samples.")
    print("Re-run the cells above to update plots.")

## 8. Fresh Start

Run the cell below to **delete** all existing measurements and run a fresh sampling pass.
Re-run all cells above afterwards to load the new data.

In [ ]:
import shutil
import subprocess

# Delete existing data
if output_dir.exists():
    shutil.rmtree(output_dir)
    print(f"Deleted {output_dir.resolve()}")
output_dir.mkdir(parents=True, exist_ok=True)

# Run fresh sampling
script = PACKAGE_ROOT / "scripts" / "workspace_analysis" / "workspace_sample.py"
cmd = [
    "conda", "run", "-n", "env_isaaclab", 
    "python3", str(script.resolve()),
    "--robot", ROBOT,
    "--num_samples", str(DENSIFY_SAMPLES),
    "--num_envs", str(DENSIFY_NUM_ENVS),
    "--output_dir", str(output_dir.resolve()),
    "--mount_direction", mount_direction,
    "--headless",
]
if MOUNT_HEIGHT is not None:
    cmd += ["--mount_height", str(MOUNT_HEIGHT)]

print(f"Running fresh sampling ({DENSIFY_SAMPLES:,} samples)...")
result = subprocess.run(
    cmd,
    capture_output=True, text=True,
    timeout=DENSIFY_TIMEOUT_S,
    cwd=str(PACKAGE_ROOT.resolve()),
)

if result.returncode != 0:
    print(f"Sampling failed (exit code {result.returncode})")
    print(result.stderr[-2000:] if result.stderr else "(no stderr)")
else:
    print(result.stderr[-2000:] if result.stderr else "(done)")
    positions, yoshikawa, condition = load_workspace_data(output_dir)
    print(f"\nLoaded {len(positions):,} samples.")
    print("Re-run the cells above to update plots.")